## Basic Multi-LLM Workflows Basic Workflows

[See python original](https://github.com/anthropics/anthropic-cookbook/blob/main/patterns/agents/basic_workflows.ipynb)

This notebook demonstrates three simple multi-LLM workflows. They trade off cost or latency for potentially improved task performances:

1. **Prompt-Chaining**: Decomposes a task into sequential subtasks, where each step builds on previous results
2. **Parallelization**: Distributes independent subtasks across multiple LLMs for concurrent processing
3. **Routing**: Dynamically selects specialized LLM paths based on input characteristics

Note: These are sample implementations meant to demonstrate core concepts - not production code.


In [1]:
require_relative "notebook" # loads required gems and add some helpers for pretty printing

# TODO: this would be better if it were Instruct.default_model "claude-3-5-haiku", access_token: ENV['ANTHROPIC_KEY']
Instruct.set_default_model "claude-3-5-sonnet-latest", access_token: ENV['ANTHROPIC_API_KEY'] # set the default model


true

In [11]:
# Chain multiple LLM calls sequentially, passing results between steps.
def chain(input, prompts)
    result = input
    prompts.each_with_index do |prompt, i|
        print "\nStep #{i}: "
        # The Instruct p{} helper uses #{} for trusted input and '<%= %>' for untrusted input
        # The gen helper tells Instruct that we want the LLM to start generating at a location
        new_prompt = p.user{"#{prompt}\nInput: <%= input %>"} + gen
        result = new_prompt.call
        print result
    end
    return result
end

:chain

### Example Usage
Below are practical examples demonstrating each workflow:

1. Chain workflow for structured data extraction and formatting
2. Parallelization workflow for stakeholder impact analysis
3. Route workflow for customer support ticket handling

In [13]:
# Example 1: Chain workflow for structured data extraction and formatting
# Each step progressively transforms raw text into a formatted table

data_processing_steps = [
    <<~STEP
        Extract only the numerical values and their associated metrics from the text.
        Format each as 'value: metric' on a new line.
        Example format:
        92: customer satisfaction
        45%: revenue growth
    STEP,
    <<~STEP
        Convert all numerical values to percentages where possible.
        If not a percentage or points, convert to decimal (e.g., 92 points -> 92%).
        Keep one number per line.
        Example format:
        92%: customer satisfaction
        45%: revenue growth
    STEP,  
    <<~STEP
        Sort all lines in descending order by numerical value.
        Keep the format 'value: metric' on each line.
        Example:
        92%: customer satisfaction
        87%: employee satisfaction
    STEP,
    <<~STEP
        Format the sorted data as a markdown table with columns:
        | Metric | Value |
        |:--|--:|
        | Customer Satisfaction | 92% |
    STEP
]

report = <<~REPORT
        Q3 Performance Summary:
        Our customer satisfaction score rose to 92 points this quarter.
        Revenue grew by 45% compared to last year.
        Market share is now at 23% in our primary market.
        Customer churn decreased to 5% from 8%.
        New user acquisition cost is $43 per user.
        Product adoption rate increased to 78%.
        Employee satisfaction is at 87 points.
        Operating margin improved to 34%.
    REPORT

chain(report, data_processing_steps)

nil


Step 0: Let me help you with that step by step.

1. Extracting numerical values and metrics:
92: customer satisfaction
45%: revenue growth
23%: market share
5%: customer churn
$43: user acquisition cost
78%: product adoption rate
87: employee satisfaction
34%: operating margin

2. Converting to percentages:
92%: customer satisfaction
45%: revenue growth
23%: market share
5%: customer churn
$43: user acquisition cost
78%: product adoption rate
87%: employee satisfaction
34%: operating margin

3. Sorting in descending order:
92%: customer satisfaction
87%: employee satisfaction
78%: product adoption rate
45%: revenue growth
34%: operating margin
23%: market share
5%: customer churn
$43: user acquisition cost

4. Formatting as markdown table:
| Metric | Value |
|:--|--:|
| Customer Satisfaction | 92% |
| Employee Satisfaction | 87% |
| Product Adoption Rate | 78% |
| Revenue Growth | 45% |
| Operating Margin | 34% |
| Market Share | 23% |
| Customer Churn | 5% |
| User Acquisition Cost |